In [0]:

entries_df = spark.table("tp_entries_raw")

entries_df.write.format("delta").mode("overwrite").saveAsTable("park_entries_bronze_python")

In [0]:
spark.table("park_entries_bronze_python").count()

133

In [0]:
from pyspark.sql.functions import col, sum, when

entries_df.select(
    sum(when(col("entry_id").isNull(), 1).otherwise(0)).alias("missing_entry_ids"),
    sum(when(col("attraction_id").isNull(), 1).otherwise(0)).alias("missing_attraction_ids"),
    sum(when(col("booking_channel").isNull(), 1).otherwise(0)).alias("missing_channels"),
    sum(when((col("ticket_price").isNull()) | (col("ticket_price") <= 0), 1).otherwise(0)).alias("invalid_prices"),
    sum(when((col("wait_minutes").isNull()) | (col("wait_minutes") < 0), 1).otherwise(0)).alias("invalid_wait_times")
).show()

+-----------------+----------------------+----------------+--------------+------------------+
|missing_entry_ids|missing_attraction_ids|missing_channels|invalid_prices|invalid_wait_times|
+-----------------+----------------------+----------------+--------------+------------------+
|                0|                     0|               3|             3|                 2|
+-----------------+----------------------+----------------+--------------+------------------+



In [0]:
from pyspark.sql.functions import col, coalesce, lit, when

silver_df = entries_df.filter(
    (col("entry_id").isNotNull()) &
    (col("attraction_id").isNotNull()) &
    (col("ticket_price") > 0) &
    (col("wait_minutes") >= 0)
)

silver_df = silver_df.withColumn(
    "booking_channel",
    coalesce(col("booking_channel"), lit("Unknown"))
)

silver_df = silver_df.withColumn(
    "ticket_revenue",
    when(col("status") == "Completed", col("ticket_price")).otherwise(0)
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("park_entries_silver_python")

In [0]:
silver_df.count()

128

In [0]:
from pyspark.sql.functions import count, sum, avg, round

attraction_gold_df = silver_df.groupBy("attraction_id").agg(
    count("*").alias("total_entries"),
    sum("ticket_revenue").alias("total_revenue"),
    round(avg("wait_minutes"), 2).alias("average_wait")
)

attraction_gold_df.write.format("delta").mode("overwrite").saveAsTable("park_gold_attraction_python")

In [0]:
spark.table("park_gold_attraction_python").orderBy("attraction_id").show()

+-------------+-------------+-------------+------------+
|attraction_id|total_entries|total_revenue|average_wait|
+-------------+-------------+-------------+------------+
|          A01|           12|       4800.0|        37.5|
|          A02|           10|       3500.0|        29.0|
|          A03|           12|       4100.0|       23.33|
|          A04|           16|       5800.0|        30.0|
|          A05|           11|       5850.0|       20.45|
|          A06|           12|       5500.0|        27.5|
|          A07|           14|       5100.0|       33.93|
|          A08|           11|       4200.0|        35.0|
|          A09|           12|       3200.0|       34.17|
|          A10|           17|       6400.0|       32.35|
|          A99|            1|        400.0|        75.0|
+-------------+-------------+-------------+------------+



In [0]:
from pyspark.sql.functions import count, sum, avg, round

attractions_df = spark.table("tp_attractions")

zone_gold_df = silver_df.join(
    attractions_df,
    "attraction_id",
    "inner"
).groupBy("location.zone").agg(
    count("*").alias("total_entries"),
    sum("ticket_revenue").alias("total_revenue"),
    round(avg("wait_minutes"), 2).alias("average_wait")
).withColumnRenamed("zone", "zone")

zone_gold_df.write.format("delta").mode("overwrite").saveAsTable("park_gold_zone_python")

In [0]:
spark.table("park_gold_zone_python").orderBy("zone").show()

+-----+-------------+-------------+------------+
| zone|total_entries|total_revenue|average_wait|
+-----+-------------+-------------+------------+
| East|           39|      15400.0|        30.0|
|North|           35|      13850.0|        31.0|
|South|           26|       9200.0|       29.04|
| West|           27|      10000.0|       32.04|
+-----+-------------+-------------+------------+

